# Module 5: Pandas Fundamentals + Data Wrangling (Solution)

This notebook is a complete, runnable solution for Module 5.

You will:
- Load a CSV into a DataFrame
- Inspect shape, types, and basic stats
- Clean and transform columns
- Filter and sort rows
- Create new calculated columns
- Group and summarize
- Export cleaned data and summary outputs


## 1) Setup paths and load data

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"

csv_file = DATA_DIR / "sales_sample.csv"
df = pd.read_csv(csv_file)

df.head()


,order_id,order_date,customer,region,product,quantity,unit_price,status
0,1001,2026-01-05,Ava,East,Notebook,2,8.99,Complete
1,1002,2026-01-05,Miles,Midwest,Pen,10,1.25,Complete
2,1003,2026-01-06,Noah,West,Notebook,1,8.99,Complete
3,1004,2026-01-06,Ava,East,Backpack,1,39.00,Returned
4,1005,2026-01-07,Priya,South,Pen,5,1.25,Complete


## 1a) Pandas check
If this fails, install Pandas in your environment before continuing.

In [2]:
# Pandas should already be available in most Anaconda environments.
# If you see an import error, install it and restart your kernel.
# - conda: conda install pandas
# - pip:   pip install pandas

print("Pandas version:", pd.__version__)


Pandas version: 2.2.3


## 2) Quick inspection

In [3]:
df.shape


(8, 8)

In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    8 non-null      int64  
 1   order_date  8 non-null      object 
 2   customer    8 non-null      object 
 3   region      8 non-null      object 
 4   product     8 non-null      object 
 5   quantity    8 non-null      int64  
 6   unit_price  8 non-null      float64
 7   status      8 non-null      object 
dtypes: float64(1), int64(2), object(5)
memory usage: 640.0+ bytes


In [5]:
df.describe(include="all")


,order_id,order_date,customer,region,product,quantity,unit_price,status
count,8.00000,8,8,8,8,8.000000,8.000000,8
unique,NaN,4,4,4,4,NaN,NaN,2
top,NaN,2026-01-05,Ava,East,Notebook,NaN,NaN,Complete
freq,NaN,2,3,3,2,NaN,NaN,7
mean,1004.50000,NaN,NaN,NaN,NaN,3.500000,12.935000,NaN
std,2.44949,NaN,NaN,NaN,NaN,2.976095,16.391424,NaN
min,1001.00000,NaN,NaN,NaN,NaN,1.000000,1.250000,NaN
25%,1002.75000,NaN,NaN,NaN,NaN,1.750000,2.187500,NaN
50%,1004.50000,NaN,NaN,NaN,NaN,2.500000,5.745000,NaN
75%,1006.25000,NaN,NaN,NaN,NaN,4.250000,16.492500,NaN


## 3) Type fixes + date parsing

In [6]:
df["order_date"] = pd.to_datetime(df["order_date"])

df.dtypes


order_id               int64
order_date    datetime64[ns]
customer              object
region                object
product               object
quantity               int64
unit_price           float64
status                object
dtype: object

## 4) Create calculated fields

In [7]:
df["revenue"] = df["quantity"] * df["unit_price"]

df[["order_id","product","quantity","unit_price","revenue"]].head()


,order_id,product,quantity,unit_price,revenue
0,1001,Notebook,2,8.99,17.98
1,1002,Pen,10,1.25,12.50
2,1003,Notebook,1,8.99,8.99
3,1004,Backpack,1,39.00,39.00
4,1005,Pen,5,1.25,6.25


## 5) Filtering and sorting
We exclude returned orders to keep revenue summaries accurate.

In [8]:
df_complete = df[df["status"] == "Complete"].copy()

# Sort by revenue descending
df_complete_sorted = df_complete.sort_values(by="revenue", ascending=False)

df_complete_sorted.head()


,order_id,order_date,customer,region,product,quantity,unit_price,status,revenue
6,1007,2026-01-08,Miles,Midwest,Backpack,2,39.00,Complete,78.00
0,1001,2026-01-05,Ava,East,Notebook,2,8.99,Complete,17.98
1,1002,2026-01-05,Miles,Midwest,Pen,10,1.25,Complete,12.50
7,1008,2026-01-08,Ava,East,Marker,4,2.50,Complete,10.00
2,1003,2026-01-06,Noah,West,Notebook,1,8.99,Complete,8.99


## 6) Grouping and aggregation

In [9]:
revenue_by_region = (
    df_complete
      .groupby("region", as_index=False)
      .agg(
          orders=("order_id", "nunique"),
          units=("quantity", "sum"),
          total_revenue=("revenue", "sum"),
          avg_order_value=("revenue", "mean")
      )
      .sort_values("total_revenue", ascending=False)
)

revenue_by_region


,region,orders,units,total_revenue,avg_order_value
1,Midwest,2,12,90.50,45.250
0,East,2,6,27.98,13.990
3,West,2,4,16.49,8.245
2,South,1,5,6.25,6.250


## 7) Pivot table example

In [10]:
pivot = pd.pivot_table(
    df_complete,
    values="revenue",
    index="region",
    columns="product",
    aggfunc="sum",
    fill_value=0
)

pivot


product,Backpack,Marker,Notebook,Pen
region,,,,
East,0.0,10.0,17.98,0.00
Midwest,78.0,0.0,0.00,12.50
South,0.0,0.0,0.00,6.25
West,0.0,7.5,8.99,0.00


## 8) Export cleaned data + summary outputs

In [11]:
cleaned_path = OUTPUT_DIR / "sales_cleaned_complete.csv"
summary_path = OUTPUT_DIR / "revenue_by_region.csv"
pivot_path = OUTPUT_DIR / "revenue_pivot.csv"

df_complete_sorted.to_csv(cleaned_path, index=False)
revenue_by_region.to_csv(summary_path, index=False)
pivot.to_csv(pivot_path)

cleaned_path, summary_path, pivot_path


(WindowsPath('C:/Users/CandaceRoberts/Downloads/Python for AI/Module5_solution/Module5/output/sales_cleaned_complete.csv'),
 WindowsPath('C:/Users/CandaceRoberts/Downloads/Python for AI/Module5_solution/Module5/output/revenue_by_region.csv'),
 WindowsPath('C:/Users/CandaceRoberts/Downloads/Python for AI/Module5_solution/Module5/output/revenue_pivot.csv'))

## 9) Completion check

In [12]:
print("Input exists:", csv_file.exists())
print("Cleaned output exists:", (OUTPUT_DIR / "sales_cleaned_complete.csv").exists())
print("Summary output exists:", (OUTPUT_DIR / "revenue_by_region.csv").exists())
print("Pivot output exists:", (OUTPUT_DIR / "revenue_pivot.csv").exists())


Input exists: True
Cleaned output exists: True
Summary output exists: True
Pivot output exists: True
